In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# In VSCode, right click on train.csv --> copy path, paste here
path = "/Users/matteogiardina/Desktop/BEMACS 2/2nd semester/ML Project/Data/train.csv"
df = pd.read_csv(path)
df = df.drop(columns=["Unnamed: 0"])
print(len(df))

110930


In [3]:
def group_problem(text):
    # Convert to uppercase to catch "PLUMBING" and "Plumbing" identically
    text = str(text).upper()
    
    # 1. Noise Complaints
    if 'NOISE' in text: 
        return 'Noise'
        
    # 2. Vehicles and Parking
    elif 'PARKING' in text or 'VEHICLE' in text or 'DRIVEWAY' in text or 'TAXI' in text or 'FOR HIRE' in text: 
        return 'Vehicles & Parking'
        
    # 3. Housing and Indoor Building Issues
    elif 'HEAT' in text or 'PAINT' in text or 'DOOR' in text or 'FLOOR' in text or 'ELECTRIC' in text or 'APPLIANCE' in text or 'ELEVATOR' in text or 'MOLD' in text or 'BOILER' in text: 
        return 'Housing & Buildings'
        
    # 4. Water, Plumbing, and Sewers
    elif 'WATER' in text or 'SEWER' in text or 'PLUMBING' in text or 'LEAK' in text: 
        return 'Water & Plumbing'
        
    # 5. Street and Infrastructure
    elif 'STREET' in text or 'SIDEWALK' in text or 'HIGHWAY' in text or 'TRAFFIC' in text or 'CURB' in text or 'SIGN' in text or 'BRIDGE' in text: 
        return 'Street & Infrastructure'
        
    # 6. Sanitation and Garbage
    elif 'SANITARY' in text or 'DIRTY' in text or 'DUMPING' in text or 'COLLECTION' in text or 'DISPOSAL' in text or 'LITTER' in text or 'SWEEPING' in text: 
        return 'Sanitation & Garbage'
        
    # 7. Trees and Parks
    elif 'TREE' in text or 'PARK' in text or 'WOOD' in text or 'STUMP' in text or 'PLANT' in text: 
        return 'Trees & Parks'
        
    # 8. Animals and Pests
    elif 'ANIMAL' in text or 'DOG' in text or 'RODENT' in text or 'PIGEON' in text or 'BEE' in text: 
        return 'Animals & Pests'
        
    # 9. General / Everything Else
    else: 
        return 'Other'

# Apply the function to create a brand new, clean column
df['Problem_Grouped'] = df['Problem (formerly Complaint Type)'].apply(group_problem)

print(df["Problem_Grouped"].value_counts(normalize=True))

Problem_Grouped
Vehicles & Parking         0.251573
Noise                      0.211061
Housing & Buildings        0.138168
Other                      0.116055
Street & Infrastructure    0.098386
Sanitation & Garbage       0.081718
Water & Plumbing           0.073037
Trees & Parks              0.016317
Animals & Pests            0.013684
Name: proportion, dtype: float64


In [4]:
def group_location(text):
    text = str(text).upper()
    
    # 1. Street and Sidewalks (Catches 'Curb', 'Highway', 'Crosswalk', etc.)
    if any(word in text for word in ['STREET', 'SIDEWALK', 'CURB', 'HIGHWAY', 'ROAD', 'INTERSECTION', 'CROSSWALK', 'LANE', 'OVERPASS', 'ISLAND']):
        return 'Street/Sidewalk'
        
    # 2. Residential (Catches 'Apartment', 'House', 'Dwelling', 'Residence')
    elif any(word in text for word in ['RESIDENT', 'HOUSE', 'APT', 'APARTMENT', 'DWELLING', 'LOFT', 'HOME']):
        return 'Residential'
        
    # 3. Commercial and Business (Catches 'Store', 'Restaurant', 'Bar', 'Vendor', etc.)
    elif any(word in text for word in ['COMMERCIAL', 'STORE', 'CLUB', 'BAR', 'RESTAURANT', 'BUSINESS', 'DELI', 'BAKERY', 'FOOD', 'VENDOR', 'CATERING', 'TATTOO', 'RETAIL', 'OFFICE', 'SHOP', 'COMERCIAL']):
        return 'Commercial/Business'
        
    # 4. Public Spaces and Transit (Catches 'Park', 'Subway', 'School', 'Bus', 'Ferry')
    elif any(word in text for word in ['PARK', 'PLAYGROUND', 'GARDEN', 'SUBWAY', 'TERMINAL', 'AIRPORT', 'BUS', 'SCHOOL', 'GOVERNMENT', 'FERRY', 'SHELTER', 'POOL', 'BRIDGE', 'STATION']):
        return 'Public/Transit'
        
    # 5. Everything Else (Catches 'Mixed Use', 'Lot', 'Alley', 'Other', NaN)
    else:
        return 'Other/Mixed'

# Apply the function to create the clean column
df['Location_Grouped'] = df['Location Type'].apply(group_location)

print(df["Location_Grouped"].value_counts(normalize=True))

Location_Grouped
Street/Sidewalk        0.437348
Residential            0.339250
Other/Mixed            0.172406
Commercial/Business    0.035446
Public/Transit         0.015550
Name: proportion, dtype: float64


In [5]:
# 1. Fill nulls with 'Unknown' safely
df['Incident Zip'] = df['Incident Zip'].fillna('Unknown')

# 2. Convert to string, then safely strip ONLY the trailing '.0' 
df['Incident Zip'] = df['Incident Zip'].astype(str).str.replace(r'\.0$', '', regex=True)

# Let's verify it worked perfectly
print(df['Incident Zip'].head())

0    10011
1    10025
2    11238
3    10453
4    10016
Name: Incident Zip, dtype: str


In [6]:
# Create a new column: 1 if it has text, 0 if it is missing (NaN)
df['Is_Landmark'] = df['Landmark'].notna().astype(int)

# Let's see the new binary split!
print(df['Is_Landmark'].value_counts())

Is_Landmark
1    64765
0    46165
Name: count, dtype: int64


In [7]:
# Create a new column: 1 if it involves a TLC vehicle, 0 if it is missing (NaN)
df['Is_Taxi'] = df['Vehicle Type'].notna().astype(int)

# Let's see the new binary split!
print(df['Is_Taxi'].value_counts())

Is_Taxi
0    105558
1      5372
Name: count, dtype: int64


In [8]:
# The exact format from your screenshot: MM/DD/YYYY HH:MM:SS AM/PM
date_format = '%m/%d/%Y %I:%M:%S %p'

# Convert strings to actual pandas datetime objects
df['Created Date'] = pd.to_datetime(df['Created Date'], format=date_format, errors='coerce')
df['Closed Date'] = pd.to_datetime(df['Closed Date'], format=date_format, errors='coerce')

# Drop any rows where the Created Date was corrupted or missing
df = df.dropna(subset=['Created Date'])

# Find the absolute latest date in the entire dataset (Data Dump Date)
max_date = max(df['Created Date'].max(), df['Closed Date'].max())
print(f"   -> Dataset extracted roughly around: {max_date}")

# Calculate how old every ticket was at the exact moment the data was pulled
df['Age at Data Dump'] = max_date - df['Created Date']

# Identify open tickets that are less than 24 hours old
unknown_outcomes = df['Closed Date'].isna() & (df['Age at Data Dump'] <= pd.Timedelta(days=1))
print(f"   -> Dropping {unknown_outcomes.sum()} rows because their 24-hour window hasn't expired yet.")

# Filter out those unknown rows
df = df[~unknown_outcomes].copy()

# Calculate time to close
df['Time to Close'] = df['Closed Date'] - df['Created Date']

# Create Y: 1 if closed within 24 hours, else 0
df['Y'] = (df['Time to Close'] <= pd.Timedelta(days=1)).astype(int)


print("\n--- DONE ---")
print("Baseline Accuracy (Distribution of Y):")
print(df['Y'].value_counts(normalize=True) * 100)

   -> Dataset extracted roughly around: 2026-04-19 01:35:00
   -> Dropping 0 rows because their 24-hour window hasn't expired yet.

--- DONE ---
Baseline Accuracy (Distribution of Y):
Y
1    61.387361
0    38.612639
Name: proportion, dtype: float64


In [9]:
# Extracting numerical time features from the Created Date
df['Created_Hour'] = df['Created Date'].dt.hour           # Returns 0-23
df['Created_DayOfWeek'] = df['Created Date'].dt.dayofweek # Returns 0-6 (0=Monday, 6=Sunday)
df['Created_Month'] = df['Created Date'].dt.month         # Returns 1-12

# You can even create custom binary features!
# 1 if it's Saturday (5) or Sunday (6), else 0
df['Is_Weekend'] = df['Created_DayOfWeek'].isin([5, 6]).astype(int) 

In [10]:
features_to_drop = [
    "Unique Key",
    "Created Date",
    "Closed Date",
    "Agency Name",
    "Problem (formerly Complaint Type)",
    "Problem Detail (formerly Descriptor)",
    "Additional Details",
    "Location Type",
    "Incident Address",
    "Street Name",
    "Cross Street 1",
    "Cross Street 2",
    "Intersection Street 1",
    "Intersection Street 2",
    "Address Type",
    "City",
    "Landmark",
    "Facility Type",
    "Community Board",
    "Council District",
    "BBL",
    "Park Facility Name",
    "Park Borough",
    "Vehicle Type",
    "Taxi Company Borough",
    "Taxi Pick Up Location",
    "Bridge Highway Name",
    "Bridge Highway Direction",
    "Bridge Highway Segment",
    "Road Ramp",
    "X Coordinate (State Plane)",
    "Y Coordinate (State Plane)",
    "Latitude",
    "Longitude",
    "Location",
    "Age at Data Dump",
    "Time to Close"
]

df = df.drop(columns=features_to_drop, errors='ignore')

# Remove 'Y' and immediately stick it on the end, just for easier visualization
df['Y'] = df.pop('Y')

In [11]:
df.head()

,Agency,Incident Zip,Police Precinct,Borough,Open Data Channel Type,Problem_Grouped,Location_Grouped,Is_Landmark,Is_Taxi,Created_Hour,Created_DayOfWeek,Created_Month,Is_Weekend,Y
0,HPD,10011,Precinct 10,MANHATTAN,MOBILE,Housing & Buildings,Residential,0,0,8,1,4,0,1
1,HPD,10025,Precinct 24,MANHATTAN,ONLINE,Housing & Buildings,Residential,0,0,15,3,4,0,0
2,DSNY,11238,Precinct 88,BROOKLYN,PHONE,Sanitation & Garbage,Street/Sidewalk,1,0,12,6,4,1,0
3,DOB,10453,Precinct 46,BRONX,UNKNOWN,Other,Other/Mixed,0,0,12,4,4,0,0
4,HPD,10016,Precinct 13,MANHATTAN,PHONE,Housing & Buildings,Residential,0,0,9,3,4,0,0


In [12]:
df.columns

Index(['Agency', 'Incident Zip', 'Police Precinct', 'Borough',
       'Open Data Channel Type', 'Problem_Grouped', 'Location_Grouped',
       'Is_Landmark', 'Is_Taxi', 'Created_Hour', 'Created_DayOfWeek',
       'Created_Month', 'Is_Weekend', 'Y'],
      dtype='str')

In [13]:
len(df)

110930